In [1]:
import yfinance as yf
import requests 
from io import StringIO
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import vectorbt as vbt
import talib
import warnings

warnings.filterwarnings("ignore")

In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\bmh.mplstyle: The text.hinting_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.
In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\classic.mplstyle: The text.hinting_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.
In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\_classic_test_patch.mplstyle: The text.kerning_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.


In [3]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

response = requests.get(url, headers=headers)
response.raise_for_status()  


table = pd.read_html(StringIO(response.text))
df_tickers = table[0]


tickers = df_tickers['Symbol'].str.replace('.', '-', regex=False).tolist()


tickers = tickers[:500]

df3 = yf.download(tickers, start= "2015-05-05", end= "2026-05-05", threads = True)

try:
  close_df = df3["Close"]
  high_df = df3["High"]
  low_df = df3["Low"]
  open_df = df3["Open"]
  volume_df = df3["volume"]

except KeyError:
  
  close_df = df3.xs("Close", level=0, axis=1)
  high_df = df3.xs("High", level=0, axis=1)
  low_df = df3.xs("Low", level=0, axis=1)
  open_df = df3.xs("Open", level = 0, axis = 1)
  volume_df = df3.xs("Volume", level = 0, axis = 1)

close_df = close_df.dropna(axis=1, how="all")
high_df = high_df.dropna(axis=1, how="all")
low_df = low_df.dropna(axis=1, how="all")
open_df = open_df.dropna(axis=1, how = "all")
volume_df = volume_df.dropna(axis =1 , how = "all")

valid_cols = (
    close_df.columns.intersection(high_df.columns)
    .intersection(low_df.columns).intersection(open_df.columns).intersection(volume_df.columns)
)
close_df, high_df, low_df, open_df, volume_df = (
    close_df[valid_cols],
    high_df[valid_cols],
    low_df[valid_cols],
    open_df[valid_cols],
    volume_df[valid_cols]
)




[*****                 10%                       ]  48 of 500 completed$VYLR: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")
[********************* 43%                       ]  217 of 500 completed$HONA: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")
[**********************67%*******                ]  337 of 500 completed$FDXF: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")
[*********************100%***********************]  500 of 500 completed

3 Failed downloads:
['VYLR', 'HONA', 'FDXF']: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")


### OBV (ON Balance Volume) Strategy for the S and P 500

In [11]:
macd_dict = {}
signal_dict = {}
hist_dict = {}
rsi_dict = {}
obv_dict = {}
obv_ma_dict = {}

for ticker in close_df.columns:
    prices = close_df[ticker].to_numpy(dtype=np.float64)

    macd, signal, hist = talib.MACD(
        prices,
        fastperiod=12,
        slowperiod=26,
        signalperiod=9
    )

    rsi = talib.RSI(
        prices,
        timeperiod=14
    )

    obv = talib.OBV(close_df[ticker].to_numpy(dtype = np.float64), volume_df[ticker].to_numpy(dtype = np.float64))
    oma = pd.Series(obv, index=close_df.index).rolling(20).mean()


    macd_dict[ticker] = macd
    signal_dict[ticker] = signal
    hist_dict[ticker] = hist
    rsi_dict[ticker] = rsi
    obv_dict[ticker] = obv
    obv_ma_dict[ticker] = oma

macd_df = pd.DataFrame(macd_dict, index=close_df.index)
signal_df = pd.DataFrame(signal_dict, index=close_df.index)
hist_df = pd.DataFrame(hist_dict, index=close_df.index)
rsi_df = pd.DataFrame(rsi_dict, index = close_df.index )
obv_df = pd.DataFrame(obv_dict, index = close_df.index)
oma_df = pd.DataFrame(obv_ma_dict, index = close_df.index)

entries = (obv_df >= oma_df).shift(1).fillna(False).astype(bool)
exits = (obv_df < oma_df).shift(1).fillna(False).astype(bool)



backtest1 = vbt.Portfolio.from_signals(
    close=close_df,
    open = open_df,
    high = high_df,
    low = low_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=0.1,                         
    size_type='percent',
    cash_sharing=True,                 
    group_by=True,                     
    freq='1D',
    fees=0.001,
    sl_stop = 0.10,
       
)

print(backtest1.stats())
backtest1.plot().show()

Start                             2015-05-05 00:00:00
End                               2026-05-04 00:00:00
Period                             2766 days 00:00:00
Start Value                                  100000.0
End Value                               200965.383981
Total Return [%]                           100.965384
Benchmark Return [%]                       524.559307
Max Gross Exposure [%]                          100.0
Total Fees Paid                          92142.025137
Max Drawdown [%]                            30.642908
Max Drawdown Duration               759 days 00:00:00
Total Trades                                    93900
Total Closed Trades                             93656
Total Open Trades                                 244
Open Trade PnL                            9475.705826
Win Rate [%]                                42.868583
Best Trade [%]                             275.673235
Worst Trade [%]                            -34.679364
Avg Winning Trade [%]       

In [14]:
### OBV Strategy paired with MACD and RSI

In [13]:
entries = ((macd_df >= signal_df) & (rsi_df >= 50) & (rsi_df < 70) &(obv_df >= oma_df)).shift(1).fillna(False).astype(bool)
exits = ((macd_df < signal_df) | (rsi_df > 70) | (obv_df < oma_df)).shift(1).fillna(False).astype(bool)



backtest1 = vbt.Portfolio.from_signals(
    close=close_df,
    open = open_df,
    high = high_df,
    low = low_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=0.1,                         
    size_type='percent',
    cash_sharing=True,                 
    group_by=True,                     
    freq='1D',
    fees=0.001,
    sl_stop = 0.10,
       
)

print(backtest1.stats())
backtest1.plot().show()

Start                            2015-05-05 00:00:00
End                              2026-05-04 00:00:00
Period                            2766 days 00:00:00
Start Value                                 100000.0
End Value                              121249.868638
Total Return [%]                           21.249869
Benchmark Return [%]                      524.559307
Max Gross Exposure [%]                     99.999991
Total Fees Paid                        113071.343151
Max Drawdown [%]                           28.324485
Max Drawdown Duration             1123 days 00:00:00
Total Trades                                   79155
Total Closed Trades                            79020
Total Open Trades                                135
Open Trade PnL                            174.107014
Win Rate [%]                               46.016198
Best Trade [%]                            108.627884
Worst Trade [%]                           -31.391328
Avg Winning Trade [%]                        3

### OBV only for entry confirmation with bullish confirmation using MACD & RSI

In [12]:
entries = ((macd_df >= signal_df) & (rsi_df >= 50) & (rsi_df < 70) &(obv_df >= oma_df)).shift(1).fillna(False).astype(bool)
exits = ((macd_df < signal_df) | (rsi_df > 70)).shift(1).fillna(False).astype(bool)



backtest1 = vbt.Portfolio.from_signals(
    close=close_df,
    open = open_df,
    high = high_df,
    low = low_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=0.1,                         
    size_type='percent',
    cash_sharing=True,                 
    group_by=True,                     
    freq='1D',
    fees=0.001,
    sl_stop = 0.10,
       
)

print(backtest1.stats())
backtest1.plot().show()

Start                            2015-05-05 00:00:00
End                              2026-05-04 00:00:00
Period                            2766 days 00:00:00
Start Value                                 100000.0
End Value                              177969.170272
Total Return [%]                            77.96917
Benchmark Return [%]                      524.559307
Max Gross Exposure [%]                         100.0
Total Fees Paid                         93437.329355
Max Drawdown [%]                           27.293325
Max Drawdown Duration              582 days 00:00:00
Total Trades                                   62376
Total Closed Trades                            62206
Total Open Trades                                170
Open Trade PnL                            260.793196
Win Rate [%]                               46.662701
Best Trade [%]                            115.774014
Worst Trade [%]                           -31.391328
Avg Winning Trade [%]                       4.